# Analiza hantavirusa primenom metoda istraživanja podataka
## Tema 1, Zadatak 1.2 — Klasterovanje nad karakteristikama ponavljajućih sekvenci

Za četiri vrste hantavirusa (Hantaan, Dobrava-Belgrade, Puumala, Sin Nombre) analiziraju se
ponavljajuće sekvence (repeats) u aminokiselinskim sekvencama GPC (glycoprotein precursor) i N
(nucleocapsid protein), sa ciljem primene klasterovanja nad izvedenim karakteristikama i
poređenja dobijenih klastera sa stvarnom taksonomskom pripadnošću vrsti.

# 1. Uvod i konfiguracija

Učitavanje biblioteka, definisanje putanja i mapiranje preuzetih FASTA fajlova
(NCBI Virus baza) na (vrsta, kompletnost).

In [1]:
import re
import os
import subprocess
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from Bio import SeqIO

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score

DATA_RAW = Path("data/raw")
DATA_FILTERED = Path("data/filtered")
REPEAT_OUTPUT_DIR = Path("repeat_output")
FIGURES_DIR = Path("figures")

DATA_FILTERED.mkdir(parents=True, exist_ok=True)
REPEAT_OUTPUT_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)


# 1. Uvod i konfiguracija (nastavak)

Mapiranje preuzetih FASTA fajlova (NCBI Virus baza) na (vrsta, kompletnost), na osnovu
broja sekvenci po fajlu i sadržaja headera (potvrđeno ranijom inspekcijom).

In [2]:
FILE_SPECIES_MAP = {
    "hantaan_all.fasta": ("hantanense", "all"),
    "hantaan_complete.fasta": ("hantanense", "complete"),
    "dobrava_all.fasta": ("dobravaense", "all"),
    "dobrava_complete.fasta": ("dobravaense", "complete"),
    "puumala_all.fasta": ("puumalaense", "all"),
    "puumala_complete.fasta": ("puumalaense", "complete"),
    "sinnombre_all.fasta": ("sinnombreense", "all"),
    "sinnombre_complete.fasta": ("sinnombreense", "complete"),
}

for fname in FILE_SPECIES_MAP:
    path = DATA_RAW / fname
    assert path.exists(), f"Fajl ne postoji: {path}"
print("Svi fajlovi pronađeni.")


Svi fajlovi pronađeni.


# 2. Klasifikacija proteina (GPC / N)

Opisi proteina u NCBI Virus bazi nisu uniformni (tipfeleri, sinonimi, različiti stilovi
anotacije). Prvo se definiše osnovna, konzervativna klasifikacija — sekvenca se
klasifikuje kao:

- **N (nucleocapsid protein)** — opis sadrži "nucleocapsid", "nucleoprotein" ili njihove
  uobičajene tipfelere.
- **GPC (glycoprotein precursor)** — opis eksplicitno pominje ceo prekursor
  ("glycoprotein precursor", "M polyprotein", "GPC"...). Pojedinačne podjedinice
  (G1, G2, Gn, Gc) su svesno isključene jer predstavljaju kraće proteinske proizvode,
  ne ceo prekursor.

In [3]:
def classify_protein(description: str) -> str | None:
    d = description.lower().strip()

    n_keywords = ["nucleocapsid", "nucleoprotein", "nucleocapsin", "nucleocapside"]
    if any(k in d for k in n_keywords):
        return "N"

    gpc_keywords = [
        "glycoprotein precursor", "glycoprotein precusor", "glicoprotein precursor",
        "m polyprotein", "gpc", "g1/g2 glycoprotein precursor",
        "g1g2 glycoprotein precursor", "envelope polyprotein",
        "precursor structural polyprotein",
    ]
    if any(k in d for k in gpc_keywords):
        return "GPC"

    return None


DESC_RE = re.compile(r"\|(.*?)\s*\[")

def extract_description(fasta_header: str) -> str:
    """fasta_header je rec.description iz Bio.SeqIO (bez '>')."""
    m = DESC_RE.search(fasta_header)
    return m.group(1).strip() if m else ""


In [4]:
test_cases = [
    "nucleocapsid protein",
    "glycoprotein precursor",
    "envelope glycoprotein",
    "G1 glycoprotein, partial",
    "RNA-dependent RNA polymerase",
    "M polyprotein",
    "GPC",
]
for t in test_cases:
    print(t, "->", classify_protein(t))


nucleocapsid protein -> N
glycoprotein precursor -> GPC
envelope glycoprotein -> None
G1 glycoprotein, partial -> None
RNA-dependent RNA polymerase -> None
M polyprotein -> GPC
GPC -> GPC


In [5]:
from collections import Counter

for fname, (species, completeness) in FILE_SPECIES_MAP.items():
    counter = Counter()
    for rec in SeqIO.parse(DATA_RAW / fname, "fasta"):
        desc = extract_description(rec.description)
        label = classify_protein(desc)
        counter[label] += 1
    print(f"{fname:25s} {species:15s} {completeness:9s} -> {dict(counter)}")


hantaan_all.fasta         hantanense      all       -> {'N': 695, None: 1356, 'GPC': 146}
hantaan_complete.fasta    hantanense      complete  -> {'N': 90, None: 156, 'GPC': 2}
dobrava_all.fasta         dobravaense     all       -> {None: 319, 'GPC': 145, 'N': 311}
dobrava_complete.fasta    dobravaense     complete  -> {None: 8, 'GPC': 13, 'N': 18}
puumala_all.fasta         puumalaense     all       -> {'N': 1787, None: 1549, 'GPC': 512}
puumala_complete.fasta    puumalaense     complete  -> {'N': 56, None: 64, 'GPC': 22}
sinnombre_all.fasta       sinnombreense   all       -> {None: 197, 'GPC': 125, 'N': 130}
sinnombre_complete.fasta  sinnombreense   complete  -> {None: 10, 'GPC': 3, 'N': 6}


Primetan je problem kod `hantaan_complete` — samo 2 sekvence klasifikovane kao GPC.
Sledeći korak je da se istraži šta se krije u kategoriji `None` za sve "complete" fajlove,
da se proveri da li se tu gubi nešto relevantno.

In [6]:
none_descriptions = Counter()

for fname, (species, completeness) in FILE_SPECIES_MAP.items():
    if completeness != "complete":
        continue
    for rec in SeqIO.parse(DATA_RAW / fname, "fasta"):
        desc = extract_description(rec.description)
        if classify_protein(desc) is None:
            none_descriptions[desc] += 1

for desc, count in none_descriptions.most_common():
    print(f"{count:4d}  {desc}")


  99  RNA-dependent RNA polymerase
  75  glycoprotein
   6  RNA polymerase
   6  glycoproteins precursor
   6  NSs
   6  RNA-dependet RNA polymerase
   5  nonstructural protein
   5  small non-structural protein
   4  RNA-directed RNA polymerase L
   3  putative RNA-dependent RNA polymerase
   3  glycoprotein polyprotein precursor
   2  G1 and G2 glycoprotein
   2  polymerase
   2  RNA-dependent RNA polymerase protein
   2  viral RNA polymerase (L protein)
   1  envelope glycoprotein G2 (see comment), partial
   1  envelope glycoprotein G1 (see comment), partial
   1  putative polymerase
   1  hypothetical protein HTNVsMgp1
   1  hypothetical protein HTNVsSgp1
   1  GP
   1  envelope glycoprotein
   1  polymerase protein
   1  RdRp
   1  L protein
   1  polyprotein
   1  putative nonstructural protein


Uočena su dva odvojena nalaza:
1. Dva keyword bag-a u trenutnoj klasifikaciji — "glycoproteins precursor" (množina) i
   "glycoprotein polyprotein precursor" (drugačiji redosled reči) nisu uhvaćeni, iako
   su nedvosmisleno GPC.
2. Najveći gubitak je opis "glycoprotein" bez ikakvog dodatka (75 sekvenci) — potrebno je
   proveriti da li se ove sekvence koncentrišu baš kod hantaan_complete (gde nedostaje GPC),
   i da li predstavljaju punu dužinu proteina ili fragmente.

In [7]:
glycoprotein_only_by_species = Counter()

for fname, (species, completeness) in FILE_SPECIES_MAP.items():
    if completeness != "complete":
        continue
    for rec in SeqIO.parse(DATA_RAW / fname, "fasta"):
        desc = extract_description(rec.description)
        if desc.strip().lower() == "glycoprotein":
            glycoprotein_only_by_species[species] += 1

print(glycoprotein_only_by_species)


Counter({'hantanense': 66, 'puumalaense': 6, 'sinnombreense': 3})


In [8]:
import statistics

lengths_precursor = []
lengths_bare = []

for rec in SeqIO.parse(DATA_RAW / "hantaan_complete.fasta", "fasta"):
    desc = extract_description(rec.description).strip().lower()
    if "precursor" in desc and "glycoprotein" in desc:
        lengths_precursor.append(len(rec.seq))
    elif desc == "glycoprotein":
        lengths_bare.append(len(rec.seq))

print("precursor:", lengths_precursor)
print("bare glycoprotein - broj:", len(lengths_bare))
print("bare glycoprotein - min/max/mean:", min(lengths_bare), max(lengths_bare), statistics.mean(lengths_bare))


precursor: []
bare glycoprotein - broj: 66
bare glycoprotein - min/max/mean: 1135 1135 1135


In [9]:
lengths_confirmed_gpc = []

for fname, (species, completeness) in FILE_SPECIES_MAP.items():
    for rec in SeqIO.parse(DATA_RAW / fname, "fasta"):
        desc = extract_description(rec.description).strip().lower()
        if "precursor" in desc and "glycoprotein" in desc:
            lengths_confirmed_gpc.append((species, len(rec.seq)))

from collections import defaultdict
by_species = defaultdict(list)
for sp, l in lengths_confirmed_gpc:
    by_species[sp].append(l)

for sp, ls in by_species.items():
    print(sp, "n=", len(ls), "min/max:", min(ls), max(ls))


hantanense n= 127 min/max: 89 1135
dobravaense n= 150 min/max: 61 1135
puumalaense n= 506 min/max: 69 1148
sinnombreense n= 114 min/max: 67 1140


**Zaključak:** maksimalne dužine potvrđenih GPC precursor sekvenci (1135–1148 AK, u
zavisnosti od vrste) poklapaju se sa dužinom "golih" `glycoprotein` opisa (1135 AK kod
hantanense). Ovo potvrđuje da je "goli" opis zapravo pun GPC precursor, samo generičnije
anotiran — treba ga uključiti u GPC klasu, ali uz proveru dužine (jer minimalne vrednosti
kod "precursor" opisa, 61–89 AK, pokazuju da i eksplicitno označeni "precursor" zapisi mogu
biti parcijalni fragmenti). Provera dužine se uvodi u sledećoj sekciji.

Pre toga, treba proveriti da li se ovaj obrazac (goli opis = puna dužina) drži i kod ostalih
vrsta gde se "goli" `glycoprotein` opis javlja (puumalaense, sinnombreense).

In [10]:
for fname, (species, completeness) in FILE_SPECIES_MAP.items():
    for rec in SeqIO.parse(DATA_RAW / fname, "fasta"):
        desc = extract_description(rec.description).strip().lower()
        if desc == "glycoprotein":
            print(species, completeness, rec.id, len(rec.seq))


hantanense all YCK98854.1 1135
hantanense all XXH92413.1 1135
hantanense all XXH92412.1 1135
hantanense all XXH92411.1 1135
hantanense all XXH92410.1 1135
hantanense all XXH92409.1 1135
hantanense all XXH92408.1 1135
hantanense all XXH92407.1 1135
hantanense all XXH92406.1 1135
hantanense all XXH92404.1 1135
hantanense all XXH92403.1 1135
hantanense all XHY08806.1 1135
hantanense all XHY08805.1 1135
hantanense all XHY08804.1 1135
hantanense all XHY08803.1 1135
hantanense all XHY08802.1 1135
hantanense all XHY08801.1 1135
hantanense all XHY08800.1 1135
hantanense all XHY08799.1 1135
hantanense all XHY08798.1 1135
hantanense all XHV16825.1 1135
hantanense all XHV16824.1 1135
hantanense all XHV16823.1 1135
hantanense all XHV16822.1 1135
hantanense all XHV16821.1 1135
hantanense all XHP16135.1 1135
hantanense all XHP16134.1 1135
hantanense all XHP16133.1 1135
hantanense all XHP16132.1 1135
hantanense all XHP16131.1 1135
hantanense all XHP16130.1 1135
hantanense all XHP16129.1 1135
hantanen

**Zaključak istraživanja:**
1. Dva keyword baga treba popraviti: "glycoproteins precursor" i "glycoprotein polyprotein precursor".
2. "Goli" opis `glycoprotein` treba uključiti u GPC klasu.
3. Ni tekst opisa ("precursor", "partial") ni sam po sebi nisu pouzdan pokazatelj pune
   dužine sekvence (postoje kratki fragmenti bez reči "partial" u opisu, npr. sinnombreense
   990/985/550 AK). Zato je neophodan nezavisan filter na osnovu stvarne dužine sekvence,
   koji se uvodi u Sekciji 3.

Na osnovu ovoga, finalizuje se `classify_protein()` funkcija.

In [11]:
def classify_protein(description: str) -> str | None:
    d = description.lower().strip()

    n_keywords = ["nucleocapsid", "nucleoprotein", "nucleocapsin", "nucleocapside"]
    if any(k in d for k in n_keywords):
        return "N"

    gpc_keywords = [
        "glycoprotein precursor", "glycoprotein precusor", "glicoprotein precursor",
        "glycoproteins precursor", "glycoprotein polyprotein precursor",
        "m polyprotein", "gpc", "g1/g2 glycoprotein precursor",
        "g1g2 glycoprotein precursor", "envelope polyprotein",
        "precursor structural polyprotein",
    ]
    if any(k in d for k in gpc_keywords):
        return "GPC"

    # "goli" opis "glycoprotein" - dokazano da odgovara punoj duzini GPC-a
    # (vidi istrazivanje iznad); puna duzina se dodatno proverava u Sekciji 3
    if d == "glycoprotein":
        return "GPC"

    return None
